In [0]:
# STAGE 1: verify the repo works in this workspace
import sys
sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")

from evaluator import warmup
from bot import strategy
from train import selfplay

warmup()
params = strategy.load_params()
mean, lo, hi = selfplay.match(params, "random", 200, "check")
print(f"bb/deal vs random: {mean:+.3f}  CI [{lo:+.3f}, {hi:+.3f}]")

In [0]:
%sql
-- Match logs (uploaded CSVs): workspace.default.hands / workspace.default.actions
-- our_net is in chips; 1 bb = 2 chips. Win = our_net > 0.
SELECT opponent, seed, COUNT(*) AS deals, ROUND(AVG(our_net/2.0), 2) AS bb_per_deal,
       ROUND(AVG(CASE WHEN our_net > 0 THEN 1.0 ELSE 0 END) * 100, 1) AS win_pct
FROM workspace.default.hands
GROUP BY opponent, seed
ORDER BY opponent, seed

In [0]:
# STAGE 2: 10k labeled rows (street, hole_cards[5], board, equity) -> workspace.default.equity_raw
# mc_equity n=2000 labels; 100 small partitions; fixed seeds (reproducible).
import sys, random
sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")
import pandas as pd

STREETS = ["PREFLOP", "FLOP", "TURN", "RIVER"]
BOARD_LEN = {"PREFLOP": 0, "FLOP": 3, "TURN": 4, "RIVER": 5}
N_ROWS, MC_N = 10_000, 2_000

def label_partition(batches):
    import sys, random
    sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")
    from evaluator import warmup
    from bot.equity import mc_equity
    warmup()
    for pdf in batches:
        rows = []
        for r in pdf.itertuples(index=False):
            rng = random.Random(int(r.seed))
            deck = list(range(36)); rng.shuffle(deck)
            hole = deck[:5]
            board = deck[5:5 + BOARD_LEN[r.street]]
            eq = mc_equity(hole, board, MC_N, random.Random(int(r.seed) + 1))
            rows.append((r.street, hole, board, round(eq, 5)))
        yield pd.DataFrame(rows, columns=["street", "hole_cards", "board", "equity"])

spec = pd.DataFrame({"street": [STREETS[i % 4] for i in range(N_ROWS)],
                     "seed": range(100_000, 100_000 + N_ROWS)})
(spark.createDataFrame(spec).repartition(100)
      .mapInPandas(label_partition,
                   "street string, hole_cards array<int>, board array<int>, equity double")
      .write.mode("overwrite").format("delta")
      .saveAsTable("workspace.default.equity_raw"))

In [0]:
# STAGE 3: verify augmentations numerically, then write workspace.default.equity_aug
# Results: hole/flop shuffle EXACT (drift 0.000000, same-seed pairing);
# suit relabeling exact at omaha_score level (0/500 mismatches); MC label drift max 0.0169,
# mean +0.0012 +/- 0.0008 -> pure MC noise (floor sd 0.0071). Labels reused.
import sys, random, zlib
sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")
from evaluator import warmup, omaha_score
from bot.equity import mc_equity
import pandas as pd
from pyspark.sql import functions as F
warmup()

rng = random.Random(5)
bad = 0
for _ in range(500):  # exact check: omaha_score must be suit-symmetric
    deck = list(range(36)); rng.shuffle(deck)
    hole, board = deck[:5], deck[5:10]
    perm = [0, 1, 2, 3]; rng.shuffle(perm)
    ph = [(c >> 2) * 4 + perm[c & 3] for c in hole]
    pb = [(c >> 2) * 4 + perm[c & 3] for c in board]
    bad += omaha_score(ph, pb) != omaha_score(hole, board)
print(f"omaha_score suit-perm exact check: {bad}/500 mismatches")

def augment(batches):
    import sys, random, zlib
    import pandas as pd
    for pdf in batches:
        rows = []
        for r in pdf.itertuples(index=False):
            street, hole, board, eq = r.street, list(r.hole_cards), list(r.board), float(r.equity)
            rows.append((street, hole, board, eq, "orig"))
            rng = random.Random(zlib.crc32(bytes(hole + board)) & 0xFFFFFFFF)
            for j in range(2):  # two random suit relabelings
                perm = [0, 1, 2, 3]
                while perm == [0, 1, 2, 3]:
                    rng.shuffle(perm)
                rows.append((street,
                             [(c >> 2) * 4 + perm[c & 3] for c in hole],
                             [(c >> 2) * 4 + perm[c & 3] for c in board],
                             eq, f"suit{j}"))
            h2 = hole[:]
            rng.shuffle(h2)
            rows.append((street, h2, board, eq, "holeshuf"))
            if len(board) >= 3:
                first3 = board[:3]
                rng.shuffle(first3)
                rows.append((street, hole, first3 + board[3:], eq, "flopshuf"))
        yield pd.DataFrame(rows, columns=["street", "hole_cards", "board", "equity", "aug"])

(spark.read.table("workspace.default.equity_raw").repartition(100)
      .mapInPandas(augment,
                   "street string, hole_cards array<int>, board array<int>, equity double, aug string")
      .write.mode("overwrite").format("delta")
      .saveAsTable("workspace.default.equity_aug"))

In [0]:
# STAGE 4: linear / GBM / numpy-MLP comparison. Split at RAW level with a canonical key
# (street, rank multisets, suit-count multisets) invariant under all augmentations.
# MLP chosen: MAE 0.0450, p50 0.26ms / p95 1.76ms (GBM 0.0328 but 3.6ms/62ms latency).
import sys, random, time, zlib
sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")
import numpy as np
from collections import Counter
from itertools import combinations
from evaluator import warmup
from bot.equity import mc_equity, preflop_equity
from bot import strategy
warmup()

def featurize(hole, board):
    f = []
    ranks_h = [c >> 2 for c in hole]; suits_h = [c & 3 for c in hole]
    ranks_b = [c >> 2 for c in board]; suits_b = [c & 3 for c in board]
    for src in (ranks_h, ranks_b):
        h = [0.0] * 9
        for r in src: h[r] += 1
        f += [x / 2 for x in h]
    for src in (suits_h, suits_b):
        s = [0.0] * 4
        for x in src: s[x] += 1
        f += [x / 5 for x in s]
    rc_h, rc_b = Counter(ranks_h), Counter(ranks_b)
    f += [sum(v >= 2 for v in rc_h.values()),
          float(max(rc_h.values()) >= 3),
          sum(v >= 2 for v in rc_b.values()),
          sum(v >= 3 for v in rc_b.values()),
          float(max(rc_b.values()) >= 4 if rc_b else 0)]
    def conn(rs):
        s = set(rs)
        if 8 in s: s.add(-1)
        return max(sum(r in s for r in range(lo, lo + 5)) for lo in range(-1, 5)) / 5
    f += [conn(ranks_h), conn(ranks_h + ranks_b), sum(ranks_h) / 40, max(ranks_h) / 8]
    f += [sum(min(v, rc_b.get(r, 0)) for r, v in rc_h.items()) / 3,
          max(v * rc_b.get(r, 0) for r, v in rc_h.items()) / 4]
    sc_h, sc_b = Counter(suits_h), Counter(suits_b)
    f += [max(sc_h[s] + sc_b.get(s, 0) for s in sc_h) / 8]
    f += [{0: 0, 3: 1, 4: 2, 5: 3}[len(board)] == i for i in range(4)]
    best1 = best2 = 0.0
    straight_share = max_suit = max_window = 0.0
    fd_share = sd_share = 0.0
    nb = len(board)
    if nb >= 3:
        n_combo = n_straight = n_fd = n_sd = 0
        for i, j in combinations(range(5), 2):
            for t in combinations(range(nb), 3):
                cards = [hole[i], hole[j]] + [board[k] for k in t]
                r5 = sorted((c >> 2 for c in cards), reverse=True)
                sc = Counter(c & 3 for c in cards)
                flush_len = max(sc.values())
                m = len(set(r5))
                if m == 5:
                    is_str = r5[0] - r5[4] == 4 or tuple(r5) == (8, 3, 2, 1, 0)
                    cat = (8 if is_str else 6) if flush_len == 5 else (3 if is_str else 0)
                else:
                    cnt = Counter(r5); mx = max(cnt.values())
                    cat = 7 if mx == 4 else 5 if mx == 3 and len(cnt) == 2 else 4 if mx == 3 else 2 if len(cnt) == 3 else 1
                sc_cat = (cat + r5[0] / 9) / 9
                if sc_cat > best1: best1, best2 = sc_cat, best1
                elif sc_cat > best2: best2 = sc_cat
                s = set(r5)
                if 8 in s: s.add(-1)
                w = max(sum(r in s for r in range(lo, lo + 5)) for lo in range(-1, 5))
                n_combo += 1
                if cat >= 3: n_straight += 1
                if flush_len == 4: n_fd += 1
                if w == 4: n_sd += 1
                max_suit = max(max_suit, flush_len / 5)
                max_window = max(max_window, w / 5)
        straight_share = n_straight / n_combo; fd_share = n_fd / n_combo; sd_share = n_sd / n_combo
    f += [best1, best2, best1 - best2, straight_share, max_suit, max_window, fd_share, sd_share]
    return np.asarray(f, dtype=np.float32)

def canonic(street, hole, board):
    rh = sorted(c >> 2 for c in hole); rb = sorted(c >> 2 for c in board)
    sh = sorted(Counter(c & 3 for c in hole).values()); sb = sorted(Counter(c & 3 for c in board).values())
    return zlib.crc32(f"{street}|{rh}|{rb}|{sh}|{sb}".encode()) % 100

raw_pdf = spark.read.table("workspace.default.equity_raw").toPandas()
aug_pdf = spark.read.table("workspace.default.equity_aug").toPandas()
test_keys = {(canonic(r.street, r.hole_cards, r.board), r.street) for r in raw_pdf.itertuples()
             if canonic(r.street, r.hole_cards, r.board) < 15}
tr = aug_pdf[[(canonic(r.street, r.hole_cards, r.board), r.street) not in test_keys for r in aug_pdf.itertuples()]]
te = raw_pdf[[(canonic(r.street, r.hole_cards, r.board), r.street) in test_keys for r in raw_pdf.itertuples()]]
Xtr = np.stack([featurize(r.hole_cards, r.board) for r in tr.itertuples()])
ytr = tr.equity.to_numpy(np.float32)
Xte = np.stack([featurize(r.hole_cards, r.board) for r in te.itertuples()])
yte = te.equity.to_numpy(np.float32)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
XtrS, XtS = (Xtr - mu) / sd, (Xte - mu) / sd

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
lin = Ridge(alpha=1.0).fit(XtrS, ytr)
gbm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, max_leaf_nodes=31,
                                    min_samples_leaf=20, random_state=0).fit(XtrS, ytr)

def train_mlp(X, y, Xv, yv, dims=(32, 16), epochs=300, batch=1024, lr=1e-3, seed=7):
    rng = np.random.default_rng(seed)
    D = X.shape[1]
    Ws = [rng.normal(0, np.sqrt(2 / D), (D, dims[0])).astype(np.float32)]
    for a, b in zip(dims, list(dims[1:]) + [1]):
        Ws.append(rng.normal(0, np.sqrt(2 / a), (a, b)).astype(np.float32))
    bl = [np.zeros(w.shape[1], np.float32) for w in Ws]
    mW = [np.zeros_like(w) for w in Ws]; mB = [np.zeros_like(b) for b in bl]
    vW = [np.zeros_like(w) for w in Ws]; vB = [np.zeros_like(b) for b in bl]
    best, best_mae, step = None, 9e9, 0
    n = len(X)
    def fwd(Xb, Ws, bl):
        a = Xb
        for l, (w, b) in enumerate(zip(Ws, bl)):
            z = a @ w + b
            a = z if l == len(Ws) - 1 else np.maximum(z, 0)
        return a
    for ep in range(epochs):
        idx = rng.permutation(n)
        for lo in range(0, n, batch):
            ii = idx[lo:lo + batch]
            x, yb = X[ii], y[ii]
            acts = [x]
            for l, (w, b) in enumerate(zip(Ws, bl)):
                z = acts[-1] @ w + b
                acts.append(z if l == len(Ws) - 1 else np.maximum(z, 0))
            g = 2 * (acts[-1] - yb[:, None]) / len(yb)
            for l in range(len(Ws) - 1, -1, -1):
                gw = acts[l].T @ g; gb_ = g.sum(0)
                if l > 0:
                    g = (g @ Ws[l].T) * (acts[l] > 0)
                step += 1
                for P, mP, vP, gr in ((Ws[l], mW[l], vW[l], gw), (bl[l], mB[l], vB[l], gb_)):
                    mP *= 0.9; mP += 0.1 * gr
                    vP *= 0.999; vP += 0.001 * gr * gr
                    P -= lr * mP / (1 - 0.9 ** step) / np.sqrt(vP / (1 - 0.999 ** step) + 1e-8)
        mae = np.abs(fwd(Xv, Ws, bl).ravel() - yv).mean()
        if mae < best_mae: best_mae, best = mae, ([w.copy() for w in Ws], [b.copy() for b in bl])
    return best

nv = 3000
(Ws, bl) = train_mlp(XtrS[nv:], ytr[nv:], XtrS[:nv], ytr[:nv])
p_lin = np.clip(lin.predict(XtS), 0, 1)
p_gbm = np.clip(gbm.predict(XtS), 0, 1)
p_mlp = np.clip(fwd(XtS, Ws, bl).ravel(), 0, 1)

def mae(p, st=None):
    m = np.ones(len(te), bool) if st is None else (te.street == st).to_numpy()
    return float(np.abs(p[m] - yte[m]).mean())

import mlflow
mlflow.autolog(disable=True)
mlflow.set_experiment("/Users/<your-databricks-user>/poker_equity_model")
for name, pred in (("ridge_linear", p_lin), ("hist_gbm", p_gbm), ("numpy_mlp", p_mlp)):
    with mlflow.start_run(run_name=name):
        mlflow.log_param("train_rows", len(tr)); mlflow.log_param("test_rows", len(te))
        mlflow.log_metric("mae_test", mae(pred))
        for s in ("PREFLOP", "FLOP", "TURN", "RIVER"):
            mlflow.log_metric(f"mae_{s}", mae(pred, s))
for name, pred in (("ridge_linear", p_lin), ("hist_gbm", p_gbm), ("numpy_mlp", p_mlp)):
    print(f"{name:14s} MAE {mae(pred):.4f}  PF {mae(pred,'PREFLOP'):.4f}  FLOP {mae(pred,'FLOP'):.4f}  "
          f"TURN {mae(pred,'TURN'):.4f}  RIV {mae(pred,'RIVER'):.4f}")

In [0]:
# STAGE 5: export the chosen MLP as bot/equity_weights.npz (+ numpy-only bot/equity_model.py)
# Uses Ws/bl/mu/sd from the Stage 4 cell. Serverless quirk: np.savez directly to /Volumes fails
# (zip needs random-access writes) - save to /tmp then copy.
import sys, shutil, os, numpy as np
sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")

out = {**{f"W{l}": w for l, w in enumerate(Ws)}, **{f"b{l}": b for l, b in enumerate(bl)},
       "mu": mu, "sd": sd}
np.savez("/tmp/equity_weights.npz", **out)
repo_npz = "/Workspace/Users/<your-databricks-user>/datathon/bot/equity_weights.npz"
shutil.copy("/tmp/equity_weights.npz", repo_npz)
shutil.copy("/tmp/equity_weights.npz", "/Volumes/workspace/default/datathon/equity_weights.npz")

# verify the exported module end-to-end (bot/equity_model.py is checked into the repo)
import importlib, bot.equity_model as em
importlib.reload(em)
from bot.equity import mc_equity
preds = np.array([em.predict([int(c) for c in r.hole_cards], [int(c) for c in r.board]) for r in te.itertuples()])
print(f"exported model held-out MAE: {np.abs(preds - yte).mean():.4f} on {len(te)} rows")
ts = []
import time
for r in te.head(300).itertuples():
    t0 = time.perf_counter(); em.predict(r.hole_cards, r.board); ts.append(time.perf_counter() - t0)
print(f"latency p50={np.median(ts)*1000:.2f} ms  p95={np.percentile(ts,95)*1000:.2f} ms")